Libraries

In [ ]:
!pip install lmfit -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lmfit.models import PseudoVoigtModel
from matplotlib.ticker import AutoMinorLocator

1. File loading

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload your .xy file here

2. SPECS .xy parser

In [ ]:
def load_specs_xy(filepath):
    """
    Reads .xy files exported from SpecsLab Prodigy.
    Returns a dictionary with metadata and a DataFrame for each region.
    """
    regions = []
    current_meta = {}
    current_data = []
    in_data = False

    with open(filepath, "r") as f:
        lines = f.readlines()

    for line in lines:
        line = line.rstrip()

        if line.startswith("#"):
            content = line[1:].strip()

            # Detect new region
            if content.startswith("Region:"):
                if current_data:
                    df = pd.DataFrame(current_data, columns=["BE", "counts"])
                    regions.append({"meta": dict(current_meta), "data": df})
                    current_data = []
                current_meta["Region"] = content.split(":", 1)[1].strip()
                in_data = False

            # Capture useful metadata
            for key in ["Analysis Method", "Excitation Energy", "Pass Energy",
                        "Binding Energy", "Dwell Time", "Curves/Scan",
                        "Values/Curve", "Acquisition Date"]:
                if content.startswith(key + ":"):
                    current_meta[key] = content.split(":", 1)[1].strip()

            if "ColumnLabels" in content:
                in_data = True  # the next non-comment line contains data

        elif in_data and line.strip():
            parts = line.split()
            if len(parts) == 2:
                try:
                    current_data.append([float(parts[0]), float(parts[1])])
                except ValueError:
                    pass

    # Save last region
    if current_data:
        df = pd.DataFrame(current_data, columns=["BE", "counts"])
        regions.append({"meta": dict(current_meta), "data": df})

    return regions

# ----- Load -----
filename = list(uploaded.keys())[0]
regions = load_specs_xy(filename)

print(f"Regions found: {len(regions)}\n")
for i, r in enumerate(regions):
    meta = r["meta"]
    df = r["data"]
    print(f"[{i}] {meta.get('Region','?'):20s} | "
          f"BE: {df['BE'].max():.1f} – {df['BE'].min():.1f} eV | "
          f"Points: {len(df)}")

Use the output above to identify the indices for B 1s, C 1s and O 1s

3. Visualization

In [ ]:
n = len(regions)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 3.5*rows))
axes = axes.flatten()

for i, r in enumerate(regions):
    df = r["data"]
    axes[i].plot(df["BE"], df["counts"], "k-", lw=1)
    axes[i].set_xlim(df["BE"].max(), df["BE"].min())  # XPS: inverted axis
    axes[i].set_title(r["meta"].get("Region", f"Region {i}"), fontsize=10)
    axes[i].set_xlabel("BE (eV)", fontsize=8)
    axes[i].set_ylabel("Counts", fontsize=8)
    axes[i].xaxis.set_minor_locator(AutoMinorLocator())

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Survey — all regions", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("survey_all_regions.png", dpi=150, bbox_inches="tight")
plt.show()

4. C 1s calibration

In [ ]:
# Use the corresponding C 1s index
IDX_C1s = 3  # <-- adjust if needed

df_c = regions[IDX_C1s]["data"].copy()

# Find the C 1s peak maximum
idx_max = df_c["counts"].idxmax()
c1s_measured = df_c.loc[idx_max, "BE"]
c1s_reference = 284.8  # eV, adventitious carbon
offset = c1s_reference - c1s_measured

print(f"Measured C 1s:    {c1s_measured:.2f} eV")
print(f"Charging offset: {offset:+.2f} eV")
print(f"\nApply correction: BE_corr = BE + ({offset:+.2f})")

# Apply correction to ALL regions
for r in regions:
    r["data"]["BE_corr"] = r["data"]["BE"] + offset

print("\n✓ Correction applied to all regions.")

5. Shirley background correction and B 1s visualization

In [ ]:
def shirley_background(x, y, tol=1e-6, max_iter=100):
    if x[0] < x[-1]:
        x, y = x[::-1], y[::-1]
        flipped = True
    else:
        flipped = False
    bg = np.linspace(y[-1], y[0], len(y))
    for _ in range(max_iter):
        bg_new = np.zeros_like(y)
        total = np.trapz(y - bg, x)
        for i in range(len(x)):
            bg_new[i] = y[-1] + (y[0] - y[-1]) * np.trapz(
                y[i:] - bg[i:], x[i:]) / (total + 1e-10)
        if np.max(np.abs(bg_new - bg)) < tol:
            break
        bg = bg_new
    if flipped:
        return bg[::-1]
    return bg

# ── Load B 1s region ──
IDX_B1s = 2
df_b = regions[IDX_B1s]["data"].copy()
x_all = df_b["BE_corr"].values
y_all = df_b["counts"].values

# ── Crop only the boron-related peak region (excluding C KLL Auger) ──
# Adjust BE_MAX if the peak rises before 194 eV
BE_MIN = 185.0
BE_MAX = 198.0

mask = (x_all >= BE_MIN) & (x_all <= BE_MAX)
x = x_all[mask]
y = y_all[mask]

# ── Shirley background over the cropped region ──
bg = shirley_background(x.copy(), y.copy())
y_corr = y - bg

# ── Two-panel figure ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: cropped region + Shirley background
axes[0].plot(x, y, "k-", lw=1.5, label="Data")
axes[0].plot(x, bg, "r--", lw=1.5, label="Background Shirley")
axes[0].fill_between(x, bg, y, alpha=0.2, color="steelblue")
axes[0].set_xlim(x.max(), x.min())
axes[0].set_xlabel("Binding energy (eV)", fontsize=12)
axes[0].set_ylabel("Counts", fontsize=12)
axes[0].set_title("B 1s — cropped region", fontsize=13)
axes[0].legend()
axes[0].xaxis.set_minor_locator(AutoMinorLocator())

# Right: corrected signal
axes[1].plot(x, y_corr, "k-", lw=1.5)
axes[1].axhline(0, color="red", lw=0.8, ls="--")
axes[1].set_xlim(x.max(), x.min())
axes[1].set_xlabel("Binding energy (eV)", fontsize=12)
axes[1].set_ylabel("Counts (background-subtracted)", fontsize=12)
axes[1].set_title("B 1s — ready for deconvolution", fontsize=13)
axes[1].xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig("B1s_shirley_ok.png", dpi=200, bbox_inches="tight")
plt.show()

# ── Peak information ──
idx_max = np.argmax(y_corr)
print(f"B 1s peak maximum: {x[idx_max]:.2f} eV")
print(f"Maximum intensity (background-corrected): {y_corr[idx_max]:.1f} counts")
print(f"Analyzed region: {BE_MAX}–{BE_MIN} eV")

# ── Save variables for later cells ──
x_b1s = x
y_b1s = y_corr
print("\n✓ Variables x_b1s and y_b1s are ready for deconvolution")

6. Two-component deconvolution

In [ ]:
from lmfit.models import PseudoVoigtModel

# ── Model: two Pseudo-Voigt peaks ──
p1 = PseudoVoigtModel(prefix="p1_")  # main B-O peak
p2 = PseudoVoigtModel(prefix="p2_")  # shoulder / secondary species
model = p1 + p2
params = model.make_params()

# ── Initial parameters ──
# Peak 1: main B–O contribution (~191.5 eV)
params["p1_center"].set(value=191.5, min=190.5, max=192.5)
params["p1_sigma"].set(value=0.6, min=0.3, max=1.5)
params["p1_amplitude"].set(value=700, min=50)
params["p1_fraction"].set(value=0.3, min=0, max=1)

# Peak 2: higher-BE shoulder (~193.5 eV)
params["p2_center"].set(value=193.5, min=192.5, max=195.5)
params["p2_sigma"].set(value=0.8, min=0.3, max=2.0)
params["p2_amplitude"].set(value=200, min=10)
params["p2_fraction"].set(value=0.3, min=0, max=1)

# ── Fit ──
result = model.fit(y_b1s, params, x=x_b1s)

# ── Numerical report ──
print("=" * 55)
print("B 1s DECONVOLUTION RESULTS")
print("=" * 55)
comps = result.eval_components(x=x_b1s)
total_area = np.trapz(result.best_fit, x_b1s)

for i, (label, assignment) in enumerate([("p1_", "B–O  (–B(OH)₂)"),
                                      ("p2_", "B–O' (secondary species)")]):
    center = result.params[f"{label}center"].value
    sigma  = result.params[f"{label}sigma"].value
    fwhm   = 2 * sigma * np.sqrt(2 * np.log(2))  # approx. for Voigt
    area_i = np.trapz(comps[label], x_b1s)
    pct    = abs(area_i / total_area) * 100
    print(f"\n{assignment}")
    print(f"  Center (BE): {center:.2f} eV")
    print(f"  FWHM:        {fwhm:.2f} eV")
    print(f"  Relative area:   {pct:.1f} %")

print(f"\nFit R²: {result.rsquared:.5f}")
print(f"Chi²:      {result.chisqr:.2f}")

# ── Publication-quality figure ──
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(7, 7),
    gridspec_kw={"height_ratios": [4, 1]},
    sharex=True
)

colors = ["steelblue", "darkorange"]
assignments = ["B–O  (–B(OH)₂)", "B–O' (secondary species)"]

# Main panel
ax1.plot(x_b1s, y_b1s, "ko", ms=3, zorder=5, label="Experimental data")
ax1.plot(x_b1s, result.best_fit, "r-", lw=2, zorder=4, label="Total fit")

for i, (label, name) in enumerate([("p1_", assignments[0]),
                                       ("p2_", assignments[1])]):
    center = result.params[f"{label}center"].value
    area_i = np.trapz(comps[label], x_b1s)
    pct    = abs(area_i / total_area) * 100
    ax1.fill_between(x_b1s, comps[label], alpha=0.35, color=colors[i])
    ax1.plot(x_b1s, comps[label], "--", color=colors[i], lw=1.5,
             label=f"{name}\n{center:.2f} eV  ({pct:.1f}%)")

ax1.set_xlim(x_b1s.max() + 0.5, x_b1s.min() - 0.5)
ax1.set_ylabel("Intensity (counts)", fontsize=12)
ax1.set_title("B 1s — XPS deconvolution", fontsize=13)
ax1.legend(fontsize=9, loc="upper left")
ax1.xaxis.set_minor_locator(AutoMinorLocator())

# Residual panel
residual = y_b1s - result.best_fit
ax2.plot(x_b1s, residual, "k-", lw=1)
ax2.axhline(0, color="red", lw=1, ls="--")
ax2.fill_between(x_b1s, residual, 0, alpha=0.2, color="gray")
ax2.set_ylabel("Residual", fontsize=10)
ax2.set_xlabel("Binding energy (eV)", fontsize=12)
ax2.xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig("B1s_deconvolution_final.png", dpi=300, bbox_inches="tight")
plt.show()

7. O 1s region

In [ ]:
# ── O 1s ──
IDX_O1s = 5
df_o = regions[IDX_O1s]["data"].copy()
x_all_o = df_o["BE_corr"].values
y_all_o = df_o["counts"].values

# Crop O 1s region (typical range: 526–538 eV)
BE_MIN_O, BE_MAX_O = 526.0, 538.0
mask_o = (x_all_o >= BE_MIN_O) & (x_all_o <= BE_MAX_O)
x_o = x_all_o[mask_o]
y_o = y_all_o[mask_o]

bg_o = shirley_background(x_o.copy(), y_o.copy())
y_o_corr = y_o - bg_o

# Quick visualization to decide the number of peaks
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(x_o, y_o, "k-", lw=1.5, label="Data")
axes[0].plot(x_o, bg_o, "r--", lw=1.5, label="Shirley")
axes[0].fill_between(x_o, bg_o, y_o, alpha=0.2, color="steelblue")
axes[0].set_xlim(x_o.max(), x_o.min())
axes[0].set_xlabel("BE (eV)"); axes[0].set_ylabel("Counts")
axes[0].set_title("O 1s — cropped region"); axes[0].legend()
axes[0].xaxis.set_minor_locator(AutoMinorLocator())

axes[1].plot(x_o, y_o_corr, "k-", lw=1.5)
axes[1].axhline(0, color="red", lw=0.8, ls="--")
axes[1].set_xlim(x_o.max(), x_o.min())
axes[1].set_xlabel("BE (eV)"); axes[1].set_ylabel("Counts (background-subtracted)")
axes[1].set_title("O 1s — ready for deconvolution")
axes[1].xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig("O1s_prefit.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"O 1s maximum at: {x_o[np.argmax(y_o_corr)]:.2f} eV")

# Save
x_o1s = x_o
y_o1s = y_o_corr

8. N 1s region

In [ ]:
# ── N 1s ──
IDX_N1s = 4
df_n = regions[IDX_N1s]["data"].copy()
x_all_n = df_n["BE_corr"].values
y_all_n = df_n["counts"].values

# Crop N 1s region (typical range: 394–404 eV)
BE_MIN_N, BE_MAX_N = 394.0, 404.0
mask_n = (x_all_n >= BE_MIN_N) & (x_all_n <= BE_MAX_N)
x_n = x_all_n[mask_n]
y_n = y_all_n[mask_n]

bg_n = shirley_background(x_n.copy(), y_n.copy())
y_n_corr = y_n - bg_n

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(x_n, y_n, "k-", lw=1.5, label="Data")
axes[0].plot(x_n, bg_n, "r--", lw=1.5, label="Shirley")
axes[0].fill_between(x_n, bg_n, y_n, alpha=0.2, color="seagreen")
axes[0].set_xlim(x_n.max(), x_n.min())
axes[0].set_xlabel("BE (eV)"); axes[0].set_ylabel("Counts")
axes[0].set_title("N 1s — cropped region"); axes[0].legend()
axes[0].xaxis.set_minor_locator(AutoMinorLocator())

axes[1].plot(x_n, y_n_corr, "k-", lw=1.5)
axes[1].axhline(0, color="red", lw=0.8, ls="--")
axes[1].set_xlim(x_n.max(), x_n.min())
axes[1].set_xlabel("BE (eV)"); axes[1].set_ylabel("Counts (background-subtracted)")
axes[1].set_title("N 1s — ready for deconvolution")
axes[1].xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig("N1s_prefit.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"N 1s maximum at: {x_n[np.argmax(y_n_corr)]:.2f} eV")

x_n1s = x_n
y_n1s = y_n_corr

9. O 1s deconvolution (2 components)

In [ ]:
# ── O 1s deconvolution ──
p1_o = PseudoVoigtModel(prefix="p1_")
p2_o = PseudoVoigtModel(prefix="p2_")
model_o = p1_o + p2_o
params_o = model_o.make_params()

# Peak 1: B–OH (~532.5 eV)
params_o["p1_center"].set(value=532.5, min=531.5, max=533.5)
params_o["p1_sigma"].set(value=0.7, min=0.3, max=1.5)
params_o["p1_amplitude"].set(value=2000, min=50)
params_o["p1_fraction"].set(value=0.3, min=0, max=1)

# Peak 2: B–O–C or C=O (~531.0 eV)
params_o["p2_center"].set(value=531.0, min=530.0, max=532.0)
params_o["p2_sigma"].set(value=0.8, min=0.3, max=1.5)
params_o["p2_amplitude"].set(value=500, min=10)
params_o["p2_fraction"].set(value=0.3, min=0, max=1)

result_o = model_o.fit(y_o1s, params_o, x=x_o1s)
comps_o = result_o.eval_components(x=x_o1s)
total_area_o = np.trapz(result_o.best_fit, x_o1s)

# Report
print("=" * 55)
print("O 1s DECONVOLUTION RESULTS")
print("=" * 55)
assign_o = ["B–OH  (–B(OH)₂)", "B–O–C / C=O"]
for i, label in enumerate(["p1_", "p2_"]):
    center = result_o.params[f"{label}center"].value
    sigma  = result_o.params[f"{label}sigma"].value
    fwhm   = 2 * sigma * np.sqrt(2 * np.log(2))
    area_i = np.trapz(comps_o[label], x_o1s)
    pct    = abs(area_i / total_area_o) * 100
    print(f"\n{assign_o[i]}")
    print(f"  Center (BE): {center:.2f} eV")
    print(f"  FWHM:        {fwhm:.2f} eV")
    print(f"  Relative area:   {pct:.1f} %")
print(f"\nR²: {result_o.rsquared:.5f}")

# Figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 7),
                                gridspec_kw={"height_ratios": [4, 1]},
                                sharex=True)
colors_o = ["steelblue", "darkorange"]

ax1.plot(x_o1s, y_o1s, "ko", ms=3, zorder=5, label="Experimental data")
ax1.plot(x_o1s, result_o.best_fit, "r-", lw=2, zorder=4, label="Total fit")

for i, (label, name) in enumerate(zip(["p1_", "p2_"], assign_o)):
    center = result_o.params[f"{label}center"].value
    area_i = np.trapz(comps_o[label], x_o1s)
    pct    = abs(area_i / total_area_o) * 100
    ax1.fill_between(x_o1s, comps_o[label], alpha=0.35, color=colors_o[i])
    ax1.plot(x_o1s, comps_o[label], "--", color=colors_o[i], lw=1.5,
             label=f"{name}\n{center:.2f} eV  ({pct:.1f}%)")

ax1.set_xlim(x_o1s.max() + 0.5, x_o1s.min() - 0.5)
ax1.set_ylabel("Intensity (counts)", fontsize=12)
ax1.set_title("O 1s — XPS deconvolution", fontsize=13)
ax1.legend(fontsize=9, loc="upper left")
ax1.xaxis.set_minor_locator(AutoMinorLocator())

residual_o = y_o1s - result_o.best_fit
ax2.plot(x_o1s, residual_o, "k-", lw=1)
ax2.axhline(0, color="red", lw=1, ls="--")
ax2.fill_between(x_o1s, residual_o, 0, alpha=0.2, color="gray")
ax2.set_ylabel("Residual", fontsize=10)
ax2.set_xlabel("Binding energy (eV)", fontsize=12)
ax2.xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig("O1s_deconvolution_final.png", dpi=300, bbox_inches="tight")
plt.show()

10. N 1s deconvolution (1 component)

In [ ]:
# ── N 1s deconvolution ──
p1_n = PseudoVoigtModel(prefix="p1_")
model_n = p1_n
params_n = model_n.make_params()

# Pyrrolic N from indole (~400 eV)
params_n["p1_center"].set(value=400.0, min=399.0, max=401.5)
params_n["p1_sigma"].set(value=0.7, min=0.3, max=1.5)
params_n["p1_amplitude"].set(value=3000, min=100)
params_n["p1_fraction"].set(value=0.3, min=0, max=1)

result_n = model_n.fit(y_n1s, params_n, x=x_n1s)
comps_n = result_n.eval_components(x=x_n1s)

# Report
print("=" * 55)
print("N 1s DECONVOLUTION RESULTS")
print("=" * 55)
center_n = result_n.params["p1_center"].value
sigma_n  = result_n.params["p1_sigma"].value
fwhm_n   = 2 * sigma_n * np.sqrt(2 * np.log(2))
print(f"\nPyrrolic N (indole –N–H)")
print(f"  Center (BE): {center_n:.2f} eV")
print(f"  FWHM:        {fwhm_n:.2f} eV")
print(f"  R²:          {result_n.rsquared:.5f}")

# Figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 7),
                                gridspec_kw={"height_ratios": [4, 1]},
                                sharex=True)

ax1.plot(x_n1s, y_n1s, "ko", ms=3, zorder=5, label="Experimental data")
ax1.plot(x_n1s, result_n.best_fit, "r-", lw=2, zorder=4, label="Total fit")
ax1.fill_between(x_n1s, comps_n["p1_"], alpha=0.35, color="seagreen")
ax1.plot(x_n1s, comps_n["p1_"], "--", color="seagreen", lw=1.5,
         label=f"Pyrrolic N (indole –N–H)\n{center_n:.2f} eV")

ax1.set_xlim(x_n1s.max() + 0.5, x_n1s.min() - 0.5)
ax1.set_ylabel("Intensity (counts)", fontsize=12)
ax1.set_title("N 1s — XPS deconvolution", fontsize=13)
ax1.legend(fontsize=10, loc="upper left")
ax1.xaxis.set_minor_locator(AutoMinorLocator())

residual_n = y_n1s - result_n.best_fit
ax2.plot(x_n1s, residual_n, "k-", lw=1)
ax2.axhline(0, color="red", lw=1, ls="--")
ax2.fill_between(x_n1s, residual_n, 0, alpha=0.2, color="gray")
ax2.set_ylabel("Residual", fontsize=10)
ax2.set_xlabel("Binding energy (eV)", fontsize=12)
ax2.xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig("N1s_deconvolution_final.png", dpi=300, bbox_inches="tight")
plt.show()

11. SUMMARY

In [ ]:
# ── Three-panel summary figure for reporting ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))

# Common configuration
panel_data = [
    # (x, y_corr, result, comps, colors, assignments, title)
    (x_b1s, y_b1s, result, comps,
     ["steelblue", "darkorange"],
     ["B–O (–B(OH)₂)\n191.47 eV (89.2%)", "B–O–C\n192.69 eV (10.8%)"],
     "B 1s"),
    (x_o1s, y_o1s, result_o, comps_o,
     ["steelblue", "darkorange"],
     ["B–OH (–B(OH)₂)\n532.88 eV (99.8%)", "B–O–C / C=O\n531.99 eV (0.2%)"],
     "O 1s"),
    (x_n1s, y_n1s, result_n, comps_n,
     ["seagreen"],
     ["Pyrrolic N–H (indole)\n400.25 eV (100%)"],
     "N 1s"),
]

prefixes_list = [["p1_", "p2_"], ["p1_", "p2_"], ["p1_"]]

for ax, (x, y, res, comp, cols, labels, title), prefixes in zip(
        axes, panel_data, prefixes_list):

    ax.plot(x, y, "ko", ms=2.5, zorder=5, label="Data")
    ax.plot(x, res.best_fit, "r-", lw=2, zorder=4, label="Fit")

    for prefix, color, label in zip(prefixes, cols, labels):
        ax.fill_between(x, comp[prefix], alpha=0.35, color=color)
        ax.plot(x, comp[prefix], "--", color=color, lw=1.5, label=label)

    ax.set_xlim(x.max() + 0.5, x.min() - 0.5)
    ax.set_xlabel("Binding energy (eV)", fontsize=11)
    ax.set_ylabel("Intensity (counts)", fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.legend(fontsize=7.5, loc="upper left")
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.axhline(0, color="gray", lw=0.5, ls=":")

plt.suptitle("XPS analysis — poly(5-indolylboronic acid)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("XPS_summary_figure.png", dpi=300, bbox_inches="tight")
plt.show()
print("✓ Figure saved as XPS_summary_figure.png")

12. Statistical test for B 1s using one- and two-component models

In [ ]:
from lmfit.models import PseudoVoigtModel
from scipy.stats import f as f_dist

# ── Fit with a single peak ──
p1_single = PseudoVoigtModel(prefix="p1_")
params_single = p1_single.make_params()
params_single["p1_center"].set(value=191.5, min=190.5, max=192.5)
params_single["p1_sigma"].set(value=0.7, min=0.3, max=2.0)
params_single["p1_amplitude"].set(value=700, min=50)
params_single["p1_fraction"].set(value=0.3, min=0, max=1)

result_1peak = p1_single.fit(y_b1s, params_single, x=x_b1s)

# ── Statistical comparison (F-test) ──
rss1 = result_1peak.chisqr   # residual sum of squares, 1 peak
rss2 = result.chisqr          # residual sum of squares, 2 peaks
n    = len(y_b1s)
p1_n = 4   # parameters in the 1-peak model
p2_n = 8   # parameters in the 2-peak model

F = ((rss1 - rss2) / (p2_n - p1_n)) / (rss2 / (n - p2_n))
p_value = 1 - f_dist.cdf(F, p2_n - p1_n, n - p2_n)

print("=" * 50)
print("COMPARISON OF 1-PEAK vs 2-PEAK MODELS — B 1s")
print("=" * 50)
print(f"\n1-peak model:")
print(f"  Chi²: {rss1:.2f}")
print(f"  R²:   {result_1peak.rsquared:.5f}")
print(f"  Center: {result_1peak.params['p1_center'].value:.2f} eV")

print(f"\n2-peak model:")
print(f"  Chi²: {rss2:.2f}")
print(f"  R²:   {result.rsquared:.5f}")

print(f"\nF-test:")
print(f"  F = {F:.2f}")
print(f"  p-value = {p_value:.4f}")

if p_value < 0.05:
    print("\n  → The second peak is statistically SIGNIFICANT (p < 0.05)")
    print("    Using two components is justified for reporting.")
else:
    print("\n  → The second peak is NOT statistically significant (p ≥ 0.05)")
    print("    Recommendation: report only one component.")

# ── Comparative figure ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (res, title) in zip(axes, [
    (result_1peak, "B 1s — 1 component"),
    (result,       "B 1s — 2 components")
]):
    ax.plot(x_b1s, y_b1s, "ko", ms=3, label="Data")
    ax.plot(x_b1s, res.best_fit, "r-", lw=2, label=f"Fit (R²={res.rsquared:.4f})")
    residual = y_b1s - res.best_fit
    ax.fill_between(x_b1s, residual * 0.1 + y_b1s.max() * 0.05,
                    y_b1s.max() * 0.05, alpha=0.3, color="gray",
                    label="Residual (×0.1)")
    ax.set_xlim(x_b1s.max() + 0.5, x_b1s.min() - 0.5)
    ax.set_xlabel("Binding energy (eV)", fontsize=11)
    ax.set_ylabel("Intensity (counts)", fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig("B1s_model_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

13. Final plots for reporting

In [ ]:
fig = plt.figure(figsize=(12, 9))
gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)
ax_survey = fig.add_subplot(gs[0, :])
ax_b1s    = fig.add_subplot(gs[1, 0])
ax_o1s    = fig.add_subplot(gs[1, 1])
ax_n1s    = fig.add_subplot(gs[1, 2])

# Get survey data
x_s = regions[1]["data"]["BE_corr"].values
y_s = regions[1]["data"]["counts"].values

# ── (a) Survey ──────────────────────────────────────────────
ax_survey.plot(x_s, y_s, "k-", lw=1)
ax_survey.set_xlim(1000, 100)
ax_survey.set_xlabel("Binding energy (eV)", fontsize=14)
ax_survey.set_ylabel("Intensity (a.u.)", fontsize=14)
ax_survey.xaxis.set_minor_locator(AutoMinorLocator())
ax_survey.set_yticks([])

y_min = y_s.min()
y_max = y_s.max()
y_range = y_max - y_min
ax_survey.set_ylim(y_min - y_range*0.05, y_max + y_range*0.28)

peaks = [
    ("C 1s",   284.8,  1.12,  False, "steelblue"),
    ("N 1s",   400.2,  0.82,  False, "steelblue"),
    ("O 1s",   532.9,  1.12,  False, "steelblue"),
    ("O KLL",  603.2,  0.80,  True,  "darkorange"),
    ("C KLL",  208.2,  0.65,  True,  "darkorange"),
    ("B 1s",   191.5,  0.48,  False, "steelblue"),
]

for label, be, frac, is_auger, color in peaks:
    idx = np.argmin(np.abs(x_s - be))
    y_peak = y_s[idx]
    y_text = y_min + frac * y_range
    ax_survey.plot([be, be], [y_peak, y_text * 0.97],
                   color=color, lw=0.8, ls="--", alpha=0.8)
    suffix = "*" if is_auger else ""
    ax_survey.text(be, y_text, f"{label}{suffix}",
                   ha="center", va="bottom", fontsize=11,
                   fontweight="bold", color=color)

ax_survey.text(0.02, 0.92, "(a)", transform=ax_survey.transAxes,
               fontsize=18, fontweight="bold", ha="left", va="top")

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="steelblue",  lw=1.5, ls="--", label="Photoelectron peaks"),
    Line2D([0], [0], color="darkorange", lw=1.5, ls="--", label="Auger (*)"),
]
ax_survey.legend(handles=legend_elements, fontsize=12,
                 loc="upper left", bbox_to_anchor=(0.12, 0.95), framealpha=0.8)
ax_survey.annotate("Ag Kα (2984.3 eV) — HAXPES",
                   xy=(0.02, 0.06), xycoords="axes fraction",
                   fontsize=12, color="gray",
                   bbox=dict(boxstyle="round,pad=0.3",
                             facecolor="white", alpha=0.7))

# ── (b) B 1s ────────────────────────────────────────────────
comps_b = result.eval_components(x=x_b1s)

ax_b1s.plot(x_b1s, y_b1s, "ko", ms=2.5, zorder=5)
ax_b1s.plot(x_b1s, result.best_fit, "r-", lw=2, zorder=4)

for prefix, color in zip(["p1_", "p2_"], ["steelblue", "darkorange"]):
    ax_b1s.fill_between(x_b1s, comps_b[prefix], alpha=0.4, color=color)
    ax_b1s.plot(x_b1s, comps_b[prefix], "-", color=color, lw=1.5)

ax_b1s.set_xlim(x_b1s.max()+0.3, x_b1s.min()-0.3)
ax_b1s.set_xlabel("Binding energy (eV)", fontsize=14)
ax_b1s.set_ylabel("Intensity (a.u.)", fontsize=14)
ax_b1s.set_yticks([])
ax_b1s.xaxis.set_minor_locator(AutoMinorLocator())
ax_b1s.text(0.06, 0.92, "(b)", transform=ax_b1s.transAxes,
            fontsize=18, fontweight="bold", ha="left", va="top")
ax_b1s.text(0.94, 0.92, "B 1s", transform=ax_b1s.transAxes,
            fontsize=18, fontweight="bold", ha="right", va="top")

# ── (c) O 1s ────────────────────────────────────────────────
comps_o2 = result_o.eval_components(x=x_o1s)

ax_o1s.plot(x_o1s, y_o1s, "ko", ms=2.5, zorder=5)
ax_o1s.plot(x_o1s, result_o.best_fit, "r-", lw=2, zorder=4)

for prefix, color in zip(["p1_", "p2_"], ["steelblue", "darkorange"]):
    ax_o1s.fill_between(x_o1s, comps_o2[prefix], alpha=0.4, color=color)
    ax_o1s.plot(x_o1s, comps_o2[prefix], "-", color=color, lw=1.5)

ax_o1s.set_xlim(x_o1s.max()+0.3, x_o1s.min()-0.3)
ax_o1s.set_xlabel("Binding energy (eV)", fontsize=14)
ax_o1s.set_ylabel("Intensity (a.u.)", fontsize=14)
ax_o1s.set_yticks([])
ax_o1s.xaxis.set_minor_locator(AutoMinorLocator())
ax_o1s.text(0.06, 0.92, "(c)", transform=ax_o1s.transAxes,
            fontsize=18, fontweight="bold", ha="left", va="top")
ax_o1s.text(0.94, 0.92, "O 1s", transform=ax_o1s.transAxes,
            fontsize=18, fontweight="bold", ha="right", va="top")

# ── (d) N 1s ────────────────────────────────────────────────
comps_n2 = result_n.eval_components(x=x_n1s)

ax_n1s.plot(x_n1s, y_n1s, "ko", ms=2.5, zorder=5)
ax_n1s.plot(x_n1s, result_n.best_fit, "r-", lw=2, zorder=4)
ax_n1s.fill_between(x_n1s, comps_n2["p1_"], alpha=0.4, color="seagreen")
ax_n1s.plot(x_n1s, comps_n2["p1_"], "-", color="seagreen", lw=1.5)

ax_n1s.set_xlim(x_n1s.max()+0.3, x_n1s.min()-0.3)
ax_n1s.set_xlabel("Binding energy (eV)", fontsize=14)
ax_n1s.set_ylabel("Intensity (a.u.)", fontsize=14)
ax_n1s.set_yticks([])
ax_n1s.xaxis.set_minor_locator(AutoMinorLocator())
ax_n1s.text(0.06, 0.92, "(d)", transform=ax_n1s.transAxes,
            fontsize=18, fontweight="bold", ha="left", va="top")
ax_n1s.text(0.94, 0.92, "N 1s", transform=ax_n1s.transAxes,
            fontsize=18, fontweight="bold", ha="right", va="top")

plt.savefig("XPS_final_figure.png", dpi=300, bbox_inches="tight")
plt.show()
print("✓ Final figure saved as XPS_final_figure.png")